# Codec Longform Checkpoint Compare

This notebook runs the neural codec branch in long-form mode on random songs from `Downloads/` so we can hear how errors accumulate over time.

It mirrors the diffusion long-form compare workflow, but uses the actual EnCodec + `CodecLatentTranslator` path chunk by chunk with cosine crossfade assembly.

In [ ]:
from pathlib import Path
import sys
import json
import importlib

def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / 'lab 3.1').exists() and (p / 'README.md').exists():
            return p
    raise RuntimeError('Could not locate repo root from current notebook cwd.')

REPO = find_repo_root()
SCRIPT_DIR = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import codec_longform_compare as clc
importlib.reload(clc)
print(REPO)

In [ ]:
cfg = clc.CodecLongformCompareConfig(
    downloads_dir=Path.home() / 'Downloads',
    run_dir=None,          # None = auto-pick newest codec run (overnight codec first)
    n_songs=2,
    targets_per_song=2,
    source_seconds=45.0,
    chunk_seconds=5.0,
    overlap_seconds=0.5,
    style_mode='mix',
    mix_alpha=0.35,
    device='auto',
    seed=328,
)
RUN_ALL = True

run_dir, panel = clc.resolve_checkpoint_panel(cfg, labels=['stage3_latest', 'stage2_latest'])
print('Resolved run dir:   ', run_dir)
print('Checkpoint panel:    ', panel)
print('Planned output root: ', cfg.output_root / cfg.tag)

In [ ]:
jobs = clc.plan_longform_jobs(cfg)
print(f'Planned {len(jobs)} jobs')
jobs[:4]

In [ ]:
if RUN_ALL:
    summary = clc.run_compare_panel(cfg, checkpoint_labels=['stage3_latest', 'stage2_latest'])
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True to launch the codec longform compare panel.')